# SentinelGraph 🛡️: Graph-Augmented Real-Time Fraud & AML Detection
### End-to-End Pipeline: Tabular Baseline, GNN (GraphSAGE), and Model Fusion

This notebook demonstrates the end-to-end training and evaluation pipeline for **SentinelGraph**, an advanced Anti-Money Laundering (AML) and transaction fraud detection system. It combines a tabular XGBoost classifier with a GraphSAGE Graph Neural Network (GNN) to capture both transaction-level features and network-level relational structures.

## 1. Setup & Environment Configurations

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score, precision_recall_curve, auc
import xgboost as xgb
import networkx as nx
import matplotlib.pyplot as plt

# Set random seed for reproducibility
np.random.seed(42)
torch.manual_seed(42)

print("Libraries imported successfully!")
print(f"PyTorch version: {torch.__version__}")

## 2. Generate Synthetic Transaction Network (Simulation)
For demonstration purposes, we will simulate a transaction network containing benign users and coordinated laundering rings (fan-in / fan-out patterns).

In [ ]:
def generate_synthetic_data(num_users=1000, num_transactions=5000):
    # Generate user features
    users = pd.DataFrame({
        'user_id': range(num_users),
        'risk_score': np.random.beta(2, 5, size=num_users),
        'account_age_months': np.random.randint(1, 120, size=num_users),
        'balance': np.random.exponential(5000, size=num_users)
    })
    
    # Generate transactions
    tx_source = np.random.choice(num_users, size=num_transactions)
    tx_target = np.random.choice(num_users, size=num_transactions)
    
    # Ensure no self-transactions
    mask = tx_source == tx_target
    tx_target[mask] = (tx_target[mask] + 1) % num_users
    
    transactions = pd.DataFrame({
        'tx_id': range(num_transactions),
        'source_user': tx_source,
        'target_user': tx_target,
        'amount': np.random.exponential(1000, size=num_transactions),
        'timestamp': sorted(np.random.randint(0, 86400, size=num_transactions)),
        'is_fraud': 0
    })
    
    # Inject laundering patterns (coordinated fraud rings)
    # Laundering Ring Type A: One-to-Many Split (1 high risk node transfers to 10 nodes)
    suspect_source = 42
    users.loc[suspect_source, 'risk_score'] = 0.95
    targets = np.random.choice(num_users, size=15, replace=False)
    for i, t in enumerate(targets):
        transactions.loc[i, 'source_user'] = suspect_source
        transactions.loc[i, 'target_user'] = t
        transactions.loc[i, 'amount'] = 15000.0
        transactions.loc[i, 'is_fraud'] = 1
        users.loc[t, 'risk_score'] = 0.8
        
    # Laundering Ring Type B: Circular transactions (Cycle)
    cycle_nodes = [101, 102, 103, 104, 105]
    for idx in range(len(cycle_nodes)):
        next_idx = (idx + 1) % len(cycle_nodes)
        tx_idx = num_transactions - 10 + idx
        transactions.loc[tx_idx, 'source_user'] = cycle_nodes[idx]
        transactions.loc[tx_idx, 'target_user'] = cycle_nodes[next_idx]
        transactions.loc[tx_idx, 'amount'] = 8000.0
        transactions.loc[tx_idx, 'is_fraud'] = 1
        
    return users, transactions

users, txs = generate_synthetic_data()
print(f"Users shape: {users.shape}, Transactions shape: {txs.shape}")
print(f"Total Fraud Transactions: {txs['is_fraud'].sum()}")

## 3. Tabular Feature Engineering & XGBoost Baseline

In [ ]:
# Engineer transaction-level features
tx_features = txs.copy()

# Merge sender and receiver balance/age features
tx_features = tx_features.merge(users.rename(columns={'user_id': 'source_user', 'risk_score': 'src_risk', 'account_age_months': 'src_age', 'balance': 'src_bal'}), on='source_user')
tx_features = tx_features.merge(users.rename(columns={'user_id': 'target_user', 'risk_score': 'dst_risk', 'account_age_months': 'dst_age', 'balance': 'dst_bal'}), on='target_user')

features_list = ['amount', 'timestamp', 'src_risk', 'src_age', 'src_bal', 'dst_risk', 'dst_age', 'dst_bal']
X = tx_features[features_list]
y = tx_features['is_fraud']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

# Train XGBoost Baseline
xgb_model = xgb.XGBClassifier(n_estimators=100, max_depth=5, learning_rate=0.1, scale_pos_weight=10, random_state=42)
xgb_model.fit(X_train, y_train)

xgb_preds = xgb_model.predict_proba(X_test)[:, 1]
print("XGBoost Test ROC-AUC:", roc_auc_score(y_test, xgb_preds))
precision, recall, _ = precision_recall_curve(y_test, xgb_preds)
print("XGBoost Test PR-AUC:", auc(recall, precision))

## 4. Graph Construction (NetworkX & PyTorch Geometric)

In [ ]:
# Construct PyTorch Geometric graph data object
edge_index = torch.tensor([txs['source_user'].values, txs['target_user'].values], dtype=torch.long)
edge_attr = torch.tensor(txs[['amount', 'timestamp']].values, dtype=torch.float)

# Node features from users table
node_features = torch.tensor(users[['risk_score', 'account_age_months', 'balance']].values, dtype=torch.float)
y_nodes = torch.zeros(users.shape[0], dtype=torch.long)

# Label users involved in fraud transactions as high risk nodes
fraud_users = set(txs[txs['is_fraud'] == 1]['source_user']).union(set(txs[txs['is_fraud'] == 1]['target_user']))
y_nodes[list(fraud_users)] = 1

print(f"Nodes features tensor: {node_features.shape}")
print(f"Edge index shape: {edge_index.shape}")
print(f"Number of fraud nodes: {y_nodes.sum().item()}")

## 5. GraphSAGE GNN Implementation

In [ ]:
# Custom SAGEConv layer to demonstrate the mathematical neighborhood aggregator
class CustomSAGEConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(CustomSAGEConv, self).__init__()
        self.lin_src = nn.Linear(in_channels, out_channels)
        self.lin_neigh = nn.Linear(in_channels, out_channels)
        
    def forward(self, x, edge_index):
        # Aggregator: Mean aggregation of neighbors
        num_nodes = x.size(0)
        deg = torch.zeros(num_nodes, dtype=torch.float).to(x.device)
        deg.index_add_(0, edge_index[1], torch.ones(edge_index.size(1)).to(x.device))
        
        # Compute message sum
        msg_sum = torch.zeros(num_nodes, x.size(1)).to(x.device)
        msg_sum.index_add_(0, edge_index[1], x[edge_index[0]])
        
        # Mean aggregation
        deg_inv = 1.0 / torch.clamp(deg, min=1.0)
        h_neigh = msg_sum * deg_inv.unsqueeze(-1)
        
        # Update state
        out = self.lin_src(x) + self.lin_neigh(h_neigh)
        return F.relu(out)

class GraphSAGEModel(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super(GraphSAGEModel, self).__init__()
        self.conv1 = CustomSAGEConv(in_channels, hidden_channels)
        self.conv2 = CustomSAGEConv(hidden_channels, out_channels)
        
    def forward(self, x, edge_index):
        h = self.conv1(x, edge_index)
        h = F.dropout(h, p=0.2, training=self.training)
        h = self.conv2(h, edge_index)
        return h

model = GraphSAGEModel(in_channels=3, hidden_channels=16, out_channels=2)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)

# Simple semi-supervised node classification training loop
model.train()
for epoch in range(150):
    optimizer.zero_grad()
    out = model(node_features, edge_index)
    loss = F.cross_entropy(out, y_nodes)
    loss.backward()
    optimizer.step()
    if (epoch + 1) % 25 == 0:
        print(f"Epoch {epoch+1:03d} | Loss: {loss.item():.4f}")

## 6. Score Fusion & Performance Comparison

In [ ]:
# Get GNN predictions
model.eval()
with torch.no_grad():
    gnn_logits = model(node_features, edge_index)
    gnn_node_probs = F.softmax(gnn_logits, dim=-1)[:, 1].numpy()

# Map node probabilities back to transaction edges
tx_features['gnn_src_prob'] = gnn_node_probs[tx_features['source_user']]
tx_features['gnn_dst_prob'] = gnn_node_probs[tx_features['target_user']]
tx_features['gnn_prob'] = (tx_features['gnn_src_prob'] + tx_features['gnn_dst_prob']) / 2.0

# Isolate test set transactions and combine models
test_txs = tx_features.loc[X_test.index]
fused_prob = 0.5 * xgb_preds + 0.5 * test_txs['gnn_prob']

# Compare models
print("=== Performance Comparison ===")
for name, probs in [("XGBoost Tabular", xgb_preds), 
                    ("GraphSAGE GNN", test_txs['gnn_prob'].values),
                    ("Fused SentinelGraph", fused_prob)]:
    roc = roc_auc_score(y_test, probs)
    pr_prec, pr_rec, _ = precision_recall_curve(y_test, probs)
    pr = auc(pr_rec, pr_prec)
    print(f"{name:<22} | ROC-AUC: {roc:.4f} | PR-AUC: {pr:.4f}")

## 7. Fraud Subgraph Explainability & Visualization
Visualizing the local neighborhood surrounding suspicious nodes helps AML analysts understand how money flows through fraud rings.

In [ ]:
# Build NetworkX graph from synthetic transactions
G = nx.DiGraph()
for _, row in txs.head(200).iterrows():
    G.add_edge(int(row['source_user']), int(row['target_user']), weight=row['amount'], is_fraud=row['is_fraud'])

# Find localized subgraph surrounding target nodes
target_nodes = [42, 101, 102, 103]
subgraph_nodes = set(target_nodes)
for n in target_nodes:
    if n in G:
        subgraph_nodes.update(G.neighbors(n))
        subgraph_nodes.update(G.predecessors(n))
        
sub_G = G.subgraph(subgraph_nodes)

plt.figure(figsize=(10, 8))
pos = nx.spring_layout(sub_G, seed=42)

# Node colors based on whether they are in target_nodes
node_colors = ['#FF4C4C' if n in target_nodes else '#4C99FF' for n in sub_G.nodes()]

# Edge colors based on fraud labels
edge_colors = ['red' if sub_G[u][v]['is_fraud'] == 1 else 'gray' for u, v in sub_G.edges()]
edge_widths = [2.5 if sub_G[u][v]['is_fraud'] == 1 else 1.0 for u, v in sub_G.edges()]

nx.draw_networkx_nodes(sub_G, pos, node_color=node_colors, node_size=600, alpha=0.9)
nx.draw_networkx_edges(sub_G, pos, edge_color=edge_colors, width=edge_widths, arrowsize=15)
nx.draw_networkx_labels(sub_G, pos, font_size=10, font_family='sans-serif', font_color='white')

plt.title("SentinelGraph Subgraph Visualization (Suspicious Relations Ring)", fontsize=14, fontweight='bold')
plt.axis('off')
plt.show()